# Teacher data, stage `resumes`

Teacher Qwen/Qwen3-14B-GGUF:Qwen3-14B-Q4_K_M.gguf on llamacpp (tp=2). Needs GPU T4 x2, Internet, secret `hg` (HF write token). Output goes to a private HF dataset repo. On T4s the engine is llama.cpp built with CUDA for sm_75: vLLM 0.33 does not start on these GPUs (probe v1/v2).

In [ ]:
%pip install -q "distill @ git+https://github.com/MSR327/explain-distill.git@896f205"
COMMIT, STAGE, MODEL, TP, ENGINE = "896f205", "resumes", "Qwen/Qwen3-14B-GGUF:Qwen3-14B-Q4_K_M.gguf", 2, "llamacpp"

In [ ]:
# llama.cpp with CUDA for compute capability 7.5 (~10 min build)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp && cd llama.cpp && git log -1 --format=%h && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j 4 --target llama-server 2>&1 | tail -2 && ls build/bin | grep llama-server

In [ ]:
# Fail fast: HF token + write access before loading a 14B model.
import os

from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("hg")
api = HfApi()
REPO = f"{api.whoami()['name']}/explain-distill-data"
api.create_repo(REPO, repo_type="dataset", private=True, exist_ok=True)
print("HF auth ok ->", REPO)

In [ ]:
from huggingface_hub import hf_hub_download
import shutil
os.makedirs("data", exist_ok=True)
for name in []:
    shutil.copy(hf_hub_download(REPO, name, repo_type="dataset"), f"data/{name}")
    print("downloaded", name)

In [ ]:
# Smoke pass on a few items: catches a broken stack in minutes, not hours.
import subprocess, sys
def run(extra, data):
    for d in ("data", data):
        os.makedirs(d, exist_ok=True)
    for name in []:
        shutil.copy(f"data/{name}", f"{data}/{name}")
    cmd = (f"{sys.executable} -m distill.teacher {STAGE} --model {MODEL} --tp {TP} --engine {ENGINE} "
           f"--data {data} {extra}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    lines = [x for x in p.stdout.splitlines() if x.startswith("RESUMES")]
    print(*lines, sep="\n")
    if p.returncode or not lines:
        print(p.stderr[-4000:])
        raise RuntimeError(f"stage failed: {cmd}")
    return lines

run("--limit 6", "smoke")
print(open("smoke/resumes.jsonl").read()[:1500])

In [ ]:
import time
t = time.time()
run("--n-train 1000 --n-eval 120", "data")
print(f"full run {(time.time() - t) / 60:.1f} min")

In [ ]:
api.upload_file(path_or_fileobj="data/resumes.jsonl", path_in_repo="resumes.jsonl", repo_id=REPO,
                repo_type="dataset", commit_message=f"resumes ({MODEL}, code commit {COMMIT})")
print("uploaded resumes.jsonl")